In [0]:
import pandas as pd

In [0]:
%pip install snowflake-connector-python msal --quiet

In [0]:
# import snowflake.connector
# import pandas as pd

# conn = snowflake.connector.connect(
#     account       = "MCKESSON-PSAS2",
#     user          = "MASOOD.GHASEMI@MCKESSON.CA",
#     authenticator = "externalbrowser",
#     warehouse     = "PRD_PSAS_ANALYTICS_TRANSPORTATION_WH",
#     role          = "PRD_PSAS_ANALYTICS_TRANSPORTATION_FR",
# )

# result = pd.read_sql(
#     "SELECT CURRENT_USER() AS logged_in_as, "
#     "       CURRENT_ROLE()  AS role, "
#     "       CURRENT_WAREHOUSE() AS warehouse",
#     conn,
# )
# conn.close()
# print("Connection successful!")
# display(result)

In [0]:
import snowflake.connector
import pandas as pd

conn = snowflake.connector.connect(
    account       = "MCKESSON-PSAS2",
    user          = "MASOOD.GHASEMI@MCKESSON.CA",
    authenticator = "externalbrowser",
    warehouse     = "PRD_PSAS_ANALYTICS_TRANSPORTATION_WH",
    role          = "PRD_PSAS_ANALYTICS_TRANSPORTATION_FR",
    database      = "PRD_PSAS_ANALYTICS_DB",
    schema        = "GOLD_TRANSPORTATION",
)

lastmile_monthly = pd.read_sql(
    "SELECT * FROM PRD_PSAS_ANALYTICS_DB.GOLD_TRANSPORTATION.vw_tsp_lastmile_mnthly",
    conn,
)
conn.close()
lastmile_monthly.head()

In [0]:
lastmile_monthly.head()

In [0]:
import snowflake.connector
import pandas as pd

conn = snowflake.connector.connect(
    account       = "MCKESSON-PSAS2",
    user          = "MASOOD.GHASEMI@MCKESSON.CA",
    authenticator = "externalbrowser",
    warehouse     = "PRD_PSAS_ANALYTICS_TRANSPORTATION_WH",
    role          = "PRD_PSAS_ANALYTICS_TRANSPORTATION_FR",
    database      = "PRD_PSAS_ANALYTICS_DB",
    schema        = "GOLD_TRANSPORTATION",
)

lastmile_events = pd.read_sql(
    "SELECT * FROM PRD_PSAS_ANALYTICS_DB.GOLD_TRANSPORTATION.VW_TSP_LASTMILE_EVENTS",
    conn,
)
conn.close()
lastmile_events.head()

In [0]:
# Save lastmile_monthly as parquet for the Streamlit app
import os
data_dir = "/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data"
os.makedirs(data_dir, exist_ok=True)
lastmile_monthly.to_parquet(f"{data_dir}/lastmile_monthly.parquet", index=False)
print(f"Saved {len(lastmile_monthly):,} rows to data/lastmile_monthly.parquet")

In [0]:
import plotly.express as px
import plotly.graph_objects as go

# Build monthly trend from events using delivery actual datetime
trend = (
    lastmile_events
    .dropna(subset=["DLVRY_ACTL_DATETIME"])
    .assign(DATE=lambda d: d["DLVRY_ACTL_DATETIME"].dt.to_period("M").dt.to_timestamp())
    .groupby("DATE", as_index=False)
    .agg(
        LASTMILE_BASE_COST=("LASTMILE_BASE_COST", "sum"),
        LASTMILE_FUEL_COST=("LASTMILE_FUEL_COST", "sum"),
        LASTMILE_MISC_COST=("LASTMILE_MISC_COST", "sum"),
        LASTMILE_TOTAL_COST=("LASTMILE_TOTAL_COST", "sum"),
        TOTAL_ROUTES=("ROUTE_COUNT_VAL", "sum"),
        TOTAL_STOPS=("STOP_COUNT_VAL_ROUTE_LVL", "sum"),
        TOTAL_TOTES=("TOTE_COUNT_VAL_ROUTE_LVL", "sum"),
    )
    .sort_values("DATE")
)

# Stacked area chart of cost components
fig = go.Figure()
for col, name in [("LASTMILE_BASE_COST", "Base"), ("LASTMILE_FUEL_COST", "Fuel"), ("LASTMILE_MISC_COST", "Misc")]:
    fig.add_trace(go.Scatter(x=trend["DATE"], y=trend[col], mode="lines", stackgroup="cost", name=name))
fig.update_layout(title="Monthly Last-Mile Cost Trend (Stacked Components)", yaxis_title="Cost ($)", xaxis_title="Month", height=450)
fig.show()

# MoM % change in total cost
trend["MOM_PCT_CHANGE"] = trend["LASTMILE_TOTAL_COST"].pct_change() * 100
fig2 = px.bar(trend.dropna(subset=["MOM_PCT_CHANGE"]), x="DATE", y="MOM_PCT_CHANGE",
              title="Month-over-Month % Change in Total Cost",
              labels={"MOM_PCT_CHANGE": "MoM Change (%)", "DATE": "Month"},
              color="MOM_PCT_CHANGE", color_continuous_scale="RdYlGn_r", height=350)
fig2.show()

# --- Day-of-week cost pattern ---
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow = (
    lastmile_events
    .dropna(subset=["DLVRY_ACTL_DATETIME", "LASTMILE_TOTAL_COST"])
    .assign(DOW=lambda d: d["DLVRY_ACTL_DATETIME"].dt.day_name())
    .groupby("DOW", as_index=False)
    .agg(
        AVG_COST=("LASTMILE_TOTAL_COST", "mean"),
        TOTAL_COST=("LASTMILE_TOTAL_COST", "sum"),
        DELIVERIES=("JOB_ID", "count"),
    )
)
dow["DOW"] = pd.Categorical(dow["DOW"], categories=dow_order, ordered=True)
dow = dow.sort_values("DOW")

fig3 = px.bar(
    dow, x="DOW", y="AVG_COST",
    title="Average Last-Mile Cost by Day of Week",
    labels={"DOW": "Day of Week", "AVG_COST": "Avg Cost per Delivery ($)"},
    color="AVG_COST", color_continuous_scale="Blues",
    text="DELIVERIES", height=380,
)
fig3.update_traces(texttemplate="n=%{text:,}", textposition="outside")
fig3.update_layout(coloraxis_showscale=False)
fig3.show()

# --- Day-of-week split by carrier (avg cost heatmap) ---
dow_carrier = (
    lastmile_events
    .dropna(subset=["DLVRY_ACTL_DATETIME", "LASTMILE_TOTAL_COST", "CARRIER_SCAC_NAME"])
    .assign(DOW=lambda d: d["DLVRY_ACTL_DATETIME"].dt.day_name())
    .groupby(["CARRIER_SCAC_NAME", "DOW"], as_index=False)
    .agg(AVG_COST=("LASTMILE_TOTAL_COST", "mean"))
    .pivot(index="CARRIER_SCAC_NAME", columns="DOW", values="AVG_COST")
    .reindex(columns=dow_order)
    .fillna(0)
)
fig4 = px.imshow(
    dow_carrier.values,
    x=dow_order, y=dow_carrier.index.tolist(),
    color_continuous_scale="YlOrRd",
    title="Avg Cost per Delivery by Carrier & Day of Week",
    labels={"x": "Day", "y": "Carrier", "color": "Avg Cost ($)"},
    height=max(400, len(dow_carrier) * 28),
    aspect="auto",
)
fig4.show()

In [0]:
import plotly.express as px
import numpy as np

# Correlation matrix using event-level numeric columns
numeric_cols = [
    "ROUTE_COUNT_VAL", "STOP_COUNT_VAL_ROUTE_LVL", "TOTE_COUNT_VAL_ROUTE_LVL",
    "LASTMILE_BASE_COST", "LASTMILE_FUEL_COST", "LASTMILE_MISC_COST",
    "LASTMILE_TOTAL_COST", "DISTANCE_VAL", "BASE_COST", "FUEL_COST",
]
short_labels = ["Routes", "Stops", "Totes", "LM Base$", "LM Fuel$", "LM Misc$", "LM Total$", "Distance", "Base$", "Fuel$"]

corr = lastmile_events[numeric_cols].corr()

fig = px.imshow(
    corr.values,
    x=short_labels, y=short_labels,
    color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
    text_auto=".2f",
    title="Correlation Matrix — Cost & Volume Metrics",
    height=550, width=650,
)
fig.show()

# Scatter: Total Cost vs Tote Volume (event level)
fig2 = px.scatter(
    lastmile_events.query("LASTMILE_TOTAL_COST > 0 and TOTE_COUNT_VAL_ROUTE_LVL > 0"),
    x="TOTE_COUNT_VAL_ROUTE_LVL", y="LASTMILE_TOTAL_COST",
    color="CARRIER_SCAC_NAME",
    opacity=0.3, trendline="ols", trendline_scope="overall",
    trendline_color_override="black",
    labels={"TOTE_COUNT_VAL_ROUTE_LVL": "Tote Count", "LASTMILE_TOTAL_COST": "Total Cost ($)", "CARRIER_SCAC_NAME": "Carrier"},
    title="Total Cost vs Tote Volume (with OLS trendline)",
    height=400,
)
fig2.show()

In [0]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import numpy as np

# Step 1: roll up to daily totals per carrier
daily_carrier = (
    lastmile_events
    .dropna(subset=["DLVRY_ACTL_DATETIME"])
    .assign(DATE=lambda d: d["DLVRY_ACTL_DATETIME"].dt.date)
    .groupby(["CARRIER_SCAC_NAME", "DATE"], as_index=False)
    .agg(
        DAILY_TOTAL_COST=("LASTMILE_TOTAL_COST", "sum"),
        DAILY_BASE_COST=("LASTMILE_BASE_COST", "sum"),
        DAILY_FUEL_COST=("LASTMILE_FUEL_COST", "sum"),
        DAILY_MISC_COST=("LASTMILE_MISC_COST", "sum"),
        DAILY_ROUTES=("ROUTE_COUNT_VAL", "sum"),
        DAILY_STOPS=("STOP_COUNT_VAL_ROUTE_LVL", "sum"),
        DAILY_TOTES=("TOTE_COUNT_VAL_ROUTE_LVL", "sum"),
        DAILY_DISTANCE=("DISTANCE_VAL", "sum"),
    )
)

# Step 2: carrier profile = average of daily totals + active day count
carrier_profile = (
    daily_carrier
    .groupby("CARRIER_SCAC_NAME", as_index=False)
    .agg(
        AVG_DAILY_TOTAL_COST=("DAILY_TOTAL_COST", "mean"),
        AVG_DAILY_BASE_COST=("DAILY_BASE_COST", "mean"),
        AVG_DAILY_FUEL_COST=("DAILY_FUEL_COST", "mean"),
        AVG_DAILY_MISC_COST=("DAILY_MISC_COST", "mean"),
        AVG_DAILY_ROUTES=("DAILY_ROUTES", "mean"),
        AVG_DAILY_STOPS=("DAILY_STOPS", "mean"),
        AVG_DAILY_TOTES=("DAILY_TOTES", "mean"),
        AVG_DAILY_DISTANCE=("DAILY_DISTANCE", "mean"),
        ACTIVE_DAYS=("DATE", "count"),
    )
)

feature_cols = [c for c in carrier_profile.columns if c != "CARRIER_SCAC_NAME"]
X = carrier_profile[feature_cols].fillna(0).values
X_scaled = StandardScaler().fit_transform(X)

# Cosine similarity matrix
sim_matrix = cosine_similarity(X_scaled)
carrier_names = carrier_profile["CARRIER_SCAC_NAME"].tolist()

fig = px.imshow(
    sim_matrix,
    x=carrier_names, y=carrier_names,
    color_continuous_scale="Blues", zmin=0, zmax=1,
    title="Carrier Similarity (Cosine) Based on Cost Profiles",
    height=700, width=800,
)
fig.update_layout(xaxis_tickangle=45)
fig.show()

# Show top-3 most similar pairs
import itertools
pairs = []
for i, j in itertools.combinations(range(len(carrier_names)), 2):
    pairs.append((carrier_names[i], carrier_names[j], sim_matrix[i, j]))
pairs_df = pd.DataFrame(pairs, columns=["Carrier_A", "Carrier_B", "Similarity"]).sort_values("Similarity", ascending=False)
print("Top 10 most similar carrier pairs:")
display(pairs_df.head(10))

In [0]:
import snowflake.connector
import pandas as pd

conn = snowflake.connector.connect(
    account       = "MCKESSON-PSAS2",
    user          = "MASOOD.GHASEMI@MCKESSON.CA",
    authenticator = "externalbrowser",
    warehouse     = "PRD_PSAS_ANALYTICS_TRANSPORTATION_WH",
    role          = "PRD_PSAS_ANALYTICS_TRANSPORTATION_FR",
    database      = "PRD_PSAS_ANALYTICS_DB",
    schema        = "GOLD_TRANSPORTATION",
)

acct_hist = pd.read_sql(
    "SELECT * FROM PRD_PSAS_ANALYTICS_DB.GOLD_TRANSPORTATION.vw_cust_acct_hist",
    conn,
)
conn.close()
acct_hist.head()

In [0]:
acct_hist

In [0]:
import pandas as pd

df = pd.read_excel("/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/data/Shipment ALL Report Import from 2025-11-30 to 01-03.xlsx")
df.head()

In [0]:
import plotly.express as px

# LASTMILE_TOTAL_COST rolled up from Zip to State (zip-level geo heatmap needs external GeoJSON)
state_rate = (
    df.assign(**{"LASTMILE_TOTAL_COST": pd.to_numeric(df["LASTMILE_TOTAL_COST"], errors="coerce")})
      .groupby("State", as_index=False)["LASTMILE_TOTAL_COST"]
      .sum()
)

fig = px.choropleth(
    state_rate,
    locations="State",
    locationmode="USA-states",
    color="LASTMILE_TOTAL_COST",
    scope="usa",
    color_continuous_scale="YlOrRd",
    title="LASTMILE_TOTAL_COST by State (aggregated from Zip)",
)
fig.show()

In [0]:
import pgeocode

# Average daily Total Rate per Zip
z = df.assign(
    **{"Total Rate": pd.to_numeric(df["Total Rate"], errors="coerce")},
    Zip5=df["Zip"].astype(str).str.extract(r"(\d{5})", expand=False),
).dropna(subset=["Zip5"])

daily_zip = z.groupby(["Zip5", "Del Date"], as_index=False)["Total Rate"].sum()
zip_rate = (
    daily_zip.groupby("Zip5", as_index=False)["Total Rate"]
    .mean()
    .rename(columns={"Total Rate": "Avg Daily Total Rate"})
)

# Look up lat/long for each zip (offline via pgeocode)
nomi = pgeocode.Nominatim("us")
coords = nomi.query_postal_code(zip_rate["Zip5"].tolist())[
    ["postal_code", "latitude", "longitude", "place_name", "state_code"]
]
zip_geo = zip_rate.merge(
    coords, left_on="Zip5", right_on="postal_code", how="left"
).dropna(subset=["latitude", "longitude"])

fig = px.scatter_geo(
    zip_geo,
    lat="latitude",
    lon="longitude",
    color="Avg Daily Total Rate",
    size="Avg Daily Total Rate",
    scope="usa",
    color_continuous_scale="YlOrRd",
    hover_name="Zip5",
    hover_data={"place_name": True, "state_code": True, "latitude": False, "longitude": False},
    title="Average Daily Total Rate by Zip Code",
)
fig.show()

In [0]:
!pip install streamlit






In [0]:

!streamlit run app.py --server.port 8501
